# Notebook Overview — Combine and Split Metadata

## Purpose

This notebook combines preprocessed metadata from all dataset sources and creates balanced **training** and **test** metadata splits for downstream feature extraction and model development.

It standardizes metadata fields, ensures class and source balance, and produces deterministic train/test splits using fixed per-source sample counts.

---

## Inputs

Preprocessed metadata CSV files from all dataset sources:

* `metadata/preprocessed/imgn_preprocessed_metadata.csv`
* `metadata/preprocessed/coco_preprocessed_metadata.csv`
* `metadata/preprocessed/diff_preprocessed_metadata.csv`
* `metadata/preprocessed/sdxl_preprocessed_metadata.csv`
* `metadata/preprocessed/mj_preprocessed_metadata.csv`
* `metadata/preprocessed/open_preprocessed_metadata.csv`

Shared configuration:

* `src/project_config.py`

---

## Execution Model

* All dataset sources are processed together in this notebook
* Each dataset is loaded independently and validated
* Only required metadata fields are retained and standardized
* All datasets are combined into a single unified metadata table
* Each dataset is shuffled independently before splitting
* Fixed per-source counts are used to create balanced train and test subsets
* Train and test subsets are combined across all sources
* Final train and test datasets are shuffled independently
* Processing is deterministic and reproducible using a fixed random seed
* No image data is accessed or modified in this notebook

---

## Outputs

**Combined Metadata CSV**  
`metadata/splits/<combined_metadata_filename>`  
Contains all dataset records with standardized fields prior to splitting.

**Training Metadata CSV**  
`metadata/splits/<train_metadata_filename>`  
Contains balanced training records across all dataset sources.

**Test Metadata CSV**  
`metadata/splits/<test_metadata_filename>`  
Contains balanced test records across all dataset sources.

**Expected Sizes**

* Equal number of samples per dataset source in each subset  
* Balanced class distribution between real and AI images  
* Total dataset size determined by configured per-source counts  

---

### 🔷 Step 1 — Environment Setup and Configuration

- Clone the GitHub repository if needed
- Verify required repository structure and configuration file
- Add the project `src` directory to the Python path
- Dynamically import `project_config.py`
- Extract notebook-specific configuration values
- Define input and output metadata paths
- Create required output directory if it does not exist
- Verify that all required preprocessed metadata CSV files are present
- Optionally display key configuration values when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 1: Environment Setup and Configuration
# ============================================

# ============================================
# USER CONFIGURATION — EDIT THIS SECTION ONLY
# ============================================

VERBOSE = True   # User toggle (True or False)

# ============================================
# END USER CONFIGURATION
# ============================================

# -------------------------------------------------
# Import required libraries
# -------------------------------------------------
import os
import sys
import importlib.util
from pathlib import Path

# -------------------------------------------------
# Define project locations
# -------------------------------------------------
BASE_DIR = "/content/dip-ai-image-detection"
PROJECT_SRC_DIR = f"{BASE_DIR}/src"
CONFIG_FILE = f"{PROJECT_SRC_DIR}/project_config.py"

print("Initializing environment...")

# -------------------------------------------------
# Clone repository if not already present
# -------------------------------------------------
if not os.path.isdir(BASE_DIR):
    print("Cloning repository...")
    !git clone -q https://github.com/pgailinas/dip-ai-image-detection.git
else:
    print("Repository already exists. Skipping clone.")

# -------------------------------------------------
# Verify required repository structure
# -------------------------------------------------
if not os.path.isdir(PROJECT_SRC_DIR):
    raise FileNotFoundError(f"Missing directory: {PROJECT_SRC_DIR}")

if not os.path.isfile(CONFIG_FILE):
    raise FileNotFoundError(f"Missing file: {CONFIG_FILE}")

# -------------------------------------------------
# Add src directory to Python path
# -------------------------------------------------
if PROJECT_SRC_DIR not in sys.path:
    sys.path.insert(0, PROJECT_SRC_DIR)

# -------------------------------------------------
# Dynamically import project configuration
# -------------------------------------------------
spec = importlib.util.spec_from_file_location("project_config", CONFIG_FILE)
cfg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cfg)

# -------------------------------------------------
# Pull only required configuration values
# -------------------------------------------------
AI_LABEL = cfg.AI_LABEL
REAL_LABEL = cfg.REAL_LABEL

SOURCE_LABEL_MAP = cfg.SOURCE_LABEL_MAP
VALID_SOURCES = cfg.VALID_SOURCES
REAL_SOURCES = cfg.REAL_SOURCES
AI_SOURCES = cfg.AI_SOURCES

SOURCE_FOLDER_NAMES = cfg.SOURCE_FOLDER_NAMES
PREPROCESSED_METADATA_FILES = cfg.PREPROCESSED_METADATA_FILES

PREPROCESSED_METADATA_DIR = cfg.PREPROCESSED_METADATA_DIR
SPLITS_METADATA_DIR = cfg.SPLITS_METADATA_DIR

TRAIN_SUBSET = cfg.TRAIN_SUBSET
TEST_SUBSET = cfg.TEST_SUBSET

TRAIN_PER_SOURCE = cfg.TRAIN_PER_SOURCE
TEST_PER_SOURCE = cfg.TEST_PER_SOURCE
RANDOM_SEED = cfg.RANDOM_SEED

COMBINED_METADATA_FILENAME = cfg.COMBINED_METADATA_FILENAME
TRAIN_METADATA_FILENAME = cfg.TRAIN_METADATA_FILENAME
TEST_METADATA_FILENAME = cfg.TEST_METADATA_FILENAME

COMBINED_METADATA_PATH = cfg.COMBINED_METADATA_PATH
TRAIN_METADATA_PATH = cfg.TRAIN_METADATA_PATH
TEST_METADATA_PATH = cfg.TEST_METADATA_PATH

# -------------------------------------------------
# Define input and output paths
# -------------------------------------------------
INPUT_CSV_DIR = Path(PREPROCESSED_METADATA_DIR)
OUTPUT_DIR = Path(SPLITS_METADATA_DIR)

# -------------------------------------------------
# Ensure output directory exists
# -------------------------------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Verify required input metadata files
# -------------------------------------------------
SOURCE_ORDER = cfg.REAL_SOURCES + cfg.AI_SOURCES

INPUT_FILES = [
    Path(PREPROCESSED_METADATA_FILES[source])
    for source in SOURCE_ORDER
]

print("Verifying required metadata CSV files...\n")

missing_files = []

for csv_path in INPUT_FILES:
    if not csv_path.exists():
        missing_files.append(str(csv_path))

if missing_files:
    raise FileNotFoundError(
        f"Missing required preprocessed metadata files: {missing_files}"
    )

print("All required metadata CSV files are present.")
print("\nEnvironment setup complete.\n")

# -------------------------------------------------
# Optional verbose display of configuration values
# -------------------------------------------------
if VERBOSE:
    print(f"BASE_DIR                  : {cfg.BASE_DIR}")
    print(f"PROJECT_SRC_DIR           : {PROJECT_SRC_DIR}")
    print(f"PREPROCESSED_METADATA_DIR : {PREPROCESSED_METADATA_DIR}")
    print(f"SPLITS_METADATA_DIR       : {SPLITS_METADATA_DIR}")
    print(f"INPUT_CSV_DIR             : {INPUT_CSV_DIR}")
    print(f"OUTPUT_DIR                : {OUTPUT_DIR}")
    print(f"TRAIN_PER_SOURCE          : {TRAIN_PER_SOURCE}")
    print(f"TEST_PER_SOURCE           : {TEST_PER_SOURCE}")
    print(f"RANDOM_SEED               : {RANDOM_SEED}")



Initializing environment...
Cloning repository...
Verifying required metadata CSV files...

All required metadata CSV files are present.

Environment setup complete.

BASE_DIR                  : /content/dip-ai-image-detection
PROJECT_SRC_DIR           : /content/dip-ai-image-detection/src
PREPROCESSED_METADATA_DIR : /content/dip-ai-image-detection/metadata/preprocessed
SPLITS_METADATA_DIR       : /content/dip-ai-image-detection/metadata/splits
INPUT_CSV_DIR             : /content/dip-ai-image-detection/metadata/preprocessed
OUTPUT_DIR                : /content/dip-ai-image-detection/metadata/splits
TRAIN_PER_SOURCE          : 2400
TEST_PER_SOURCE           : 600
RANDOM_SEED               : 42


### 🔷 Step 2 — Define Paths

- Confirm input and output metadata directory paths
- Prepare for loading preprocessed metadata CSV files
- Optionally display path configuration when `VERBOSE=True`

---

In [ ]:
# =========================
# Step 2: Define Paths
# =========================

# -------------------------------------------------
# Import required library
# -------------------------------------------------
import pandas as pd

# -------------------------------------------------
# Confirm path configuration
# -------------------------------------------------
if VERBOSE:
    print("Paths defined.")
    print(f"INPUT_CSV_DIR = {INPUT_CSV_DIR}")
    print(f"OUTPUT_DIR    = {OUTPUT_DIR}")



Paths defined.
INPUT_CSV_DIR = /content/dip-ai-image-detection/metadata/preprocessed
OUTPUT_DIR    = /content/dip-ai-image-detection/metadata/splits


### 🔷 Step 3 — Dataset Configuration

- Define dataset configuration mapping for all sources
- Associate each dataset with its metadata CSV file and class label
- Store source identifiers for downstream processing
- Optionally display dataset configuration details when `VERBOSE=True`

---

In [ ]:
# =========================
# Step 3: Dataset Configuration
# =========================

# -------------------------------------------------
# Build dataset configuration mapping
# -------------------------------------------------
DATASET_CONFIG = {}

for source in VALID_SOURCES:
    dataset_name = SOURCE_FOLDER_NAMES[source]

    DATASET_CONFIG[dataset_name] = {
        "csv": Path(PREPROCESSED_METADATA_FILES[source]).name,
        "class_label": SOURCE_LABEL_MAP[source],
        "source_key": source,   # Reference to original source key
    }

print("Dataset configurations defined.")

# -------------------------------------------------
# Optional verbose display of dataset configuration
# -------------------------------------------------
if VERBOSE:
    for name, cfg_entry in DATASET_CONFIG.items():
        print(f"{name}:")
        print(f"  csv         : {cfg_entry['csv']}")
        print(f"  class_label : {cfg_entry['class_label']}")



Dataset configurations defined.
DiffusionDB:
  csv         : diff_preprocessed_metadata.csv
  class_label : ai
SDXL_Generated_10K:
  csv         : sdxl_preprocessed_metadata.csv
  class_label : ai
Midjourney:
  csv         : midj_preprocessed_metadata.csv
  class_label : ai
ImageNet_1K_256:
  csv         : imgn_preprocessed_metadata.csv
  class_label : rl
MS_COCO_2017:
  csv         : coco_preprocessed_metadata.csv
  class_label : rl
OpenImages:
  csv         : open_preprocessed_metadata.csv
  class_label : rl


### 🔷 Step 4 — Load and Combine Metadata

- Load preprocessed metadata CSV files for all dataset sources
- Validate required columns in each dataset
- Standardize metadata fields across datasets
- Combine all datasets into a single unified DataFrame
- Optionally display dataset statistics when `VERBOSE=True`
- Save combined metadata CSV for downstream processing

---

In [ ]:
# =========================
# Step 4: Load and Combine Metadata
# =========================

# -------------------------------------------------
# Initialize storage for per-dataset DataFrames
# -------------------------------------------------
dfs = []

BASE_COLUMNS = ["filename"]

# -------------------------------------------------
# Load and standardize each dataset metadata file
# -------------------------------------------------
for dataset_name, dataset_cfg in DATASET_CONFIG.items():
    csv_path = INPUT_CSV_DIR / dataset_cfg["csv"]

    if not csv_path.exists():
        raise FileNotFoundError(f"Missing input CSV: {csv_path}")

    df = pd.read_csv(csv_path)

    if "filename" not in df.columns:
        raise ValueError(f"'filename' column not found in {csv_path}")

    # Keep only required columns
    df = df[BASE_COLUMNS].copy()

    # Add standardized metadata fields
    df["source_dataset"] = dataset_name
    df["class_label"] = dataset_cfg["class_label"]

    dfs.append(df)

# -------------------------------------------------
# Combine all dataset DataFrames
# -------------------------------------------------
combined_df = pd.concat(dfs, ignore_index=True)

# -------------------------------------------------
# Optional verbose display of combined dataset
# -------------------------------------------------
if VERBOSE:
    print("Combined dataset shape:", combined_df.shape)

    print("\nCounts by source_dataset:")
    print(combined_df["source_dataset"].value_counts())

    print("\nCounts by class_label:")
    print(combined_df["class_label"].value_counts())

    print("\nColumns in combined dataset:")
    print(combined_df.columns.tolist())

# -------------------------------------------------
# Save combined metadata CSV
# -------------------------------------------------
combined_csv_path = Path(COMBINED_METADATA_PATH)
combined_df.to_csv(combined_csv_path, index=False)

print(f"\nSaved combined metadata to: {combined_csv_path}")



Combined dataset shape: (18000, 3)

Counts by source_dataset:
source_dataset
DiffusionDB           3000
SDXL_Generated_10K    3000
Midjourney            3000
ImageNet_1K_256       3000
MS_COCO_2017          3000
OpenImages            3000
Name: count, dtype: int64

Counts by class_label:
class_label
ai    9000
rl    9000
Name: count, dtype: int64

Columns in combined dataset:
['filename', 'source_dataset', 'class_label']

Saved combined metadata to: /content/dip-ai-image-detection/metadata/splits/combined_metadata.csv


### 🔷 Step 5 — Split, Save, and Validate Metadata

- Split each dataset source into train and test subsets using fixed per-source counts
- Shuffle each dataset independently before splitting
- Assign standardized subset labels to train and test records
- Combine all source-level splits into unified metadata tables
- Shuffle final train and test metadata independently
- Save train and test metadata CSV files
- Optionally display split counts and class/source balance when `VERBOSE=True`
- Validate that the combined, train, and test metadata CSV files were saved successfully

---

In [ ]:
# =========================
# Step 5: Split, Save, and Validate Metadata
# =========================

# -------------------------------------------------
# Initialize storage for split DataFrames
# -------------------------------------------------
split_dfs = []

# -------------------------------------------------
# Split each dataset into train/test
# -------------------------------------------------
for dataset_name, group_df in combined_df.groupby("source_dataset"):
    # Shuffle each dataset independently
    group_df = group_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    # Exact-count split
    train_df = group_df.iloc[:TRAIN_PER_SOURCE].copy()
    test_df = group_df.iloc[
        TRAIN_PER_SOURCE:TRAIN_PER_SOURCE + TEST_PER_SOURCE
    ].copy()

    # Assign subset labels
    train_df["subset"] = TRAIN_SUBSET
    test_df["subset"] = TEST_SUBSET

    split_dfs.extend([train_df, test_df])

# -------------------------------------------------
# Combine all splits
# -------------------------------------------------
split_df = pd.concat(split_dfs, ignore_index=True)

train_metadata = split_df[split_df["subset"] == TRAIN_SUBSET].copy()
test_metadata = split_df[split_df["subset"] == TEST_SUBSET].copy()

# -------------------------------------------------
# Shuffle final train and test metadata
# -------------------------------------------------
train_metadata = train_metadata.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
test_metadata = test_metadata.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# -------------------------------------------------
# Save train and test metadata CSV files
# -------------------------------------------------
train_csv_path = Path(TRAIN_METADATA_PATH)
test_csv_path = Path(TEST_METADATA_PATH)

train_metadata.to_csv(train_csv_path, index=False)
test_metadata.to_csv(test_csv_path, index=False)

# -------------------------------------------------
# Optional verbose display of split results
# -------------------------------------------------
if VERBOSE:
    print("Train shape:", train_metadata.shape)
    print("Test shape:", test_metadata.shape)

    print("\nCounts by subset:")
    print(pd.Series({
        TRAIN_SUBSET: len(train_metadata),
        TEST_SUBSET: len(test_metadata),
    }))

    print("\nCounts by subset and source_dataset:")
    print(split_df.groupby(["subset", "source_dataset"]).size())

    print("\nCounts by subset and class_label:")
    print(split_df.groupby(["subset", "class_label"]).size())

print(f"\nSaved train metadata to: {train_csv_path}")
print(f"Saved test metadata to: {test_csv_path}")

# -------------------------------------------------
# Validate saved CSV files
# -------------------------------------------------
print("\nValidation of saved CSV files:")

for csv_path in [
    Path(COMBINED_METADATA_PATH),
    Path(TRAIN_METADATA_PATH),
    Path(TEST_METADATA_PATH),
]:
    if not csv_path.exists():
        print(f"Missing: {csv_path}")
    else:
        df = pd.read_csv(csv_path)
        print(f"{csv_path.name}: {df.shape}")



Train shape: (14400, 4)
Test shape: (3600, 4)

Counts by subset:
train    14400
test      3600
dtype: int64

Counts by subset and source_dataset:
subset  source_dataset    
test    DiffusionDB            600
        ImageNet_1K_256        600
        MS_COCO_2017           600
        Midjourney             600
        OpenImages             600
        SDXL_Generated_10K     600
train   DiffusionDB           2400
        ImageNet_1K_256       2400
        MS_COCO_2017          2400
        Midjourney            2400
        OpenImages            2400
        SDXL_Generated_10K    2400
dtype: int64

Counts by subset and class_label:
subset  class_label
test    ai             1800
        rl             1800
train   ai             7200
        rl             7200
dtype: int64

Saved train metadata to: /content/dip-ai-image-detection/metadata/splits/train_metadata.csv
Saved test metadata to: /content/dip-ai-image-detection/metadata/splits/test_metadata.csv

Validation of saved CSV files: